## AST Parsing for different code files needed to be done

We are going to use the ***tree-sitter*** module to get going


In [2]:
from pygments.lexers import get_lexer_for_filename
from pygments.util import ClassNotFound
from pathlib import Path
from tree_sitter_language_pack import get_parser, get_language
from tree_sitter import Query, Node, QueryCursor, Tree
from typing import List, Dict, Any, Tuple, Optional

In [3]:
def extension_to_language_name(file_name: Path, get_full_name: bool = False) -> str:
    try:
        lexer = get_lexer_for_filename(file_name)
        if get_full_name:
            return lexer.name
        return lexer.aliases[0] if lexer.aliases else "text"
    except ClassNotFound:
        return "unknown"  # Fallback agar extension parse na ho paaye

In [4]:
files = [
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.cpp",
    "/home/user/Documents/Project_5/backend/experiments/data/code/oops.cpp",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.cs",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.go",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.java",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.js",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.md",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.py",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.rs",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.scala",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.tsx",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.txt",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.vue",
    "/home/user/Documents/Project_5/backend/experiments/data/code/palindrome.html",
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.csv",
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.doc",
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.docx",
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.ppt",
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.pptx",
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.xls",
    "/home/user/Documents/Project_5/backend/experiments/data/code/documents/test.xlsx"
]

In [5]:
aliases = []
for file in files:
    aliases.append(extension_to_language_name(Path(file), False))

### Trying out the tree-sitter 

In [6]:
KNOWN_MISMATCHES = {"csharp": "c_sharp", "text": None}


def normalize_alias_to_treesitter(alias: str) -> str:
    if alias in KNOWN_MISMATCHES:
        return KNOWN_MISMATCHES[alias]

    return alias.lower().replace("-", "_")

In [7]:
for alias in aliases:
    print(normalize_alias_to_treesitter(alias))

cpp
cpp
c_sharp
go
java
javascript
markdown
python
rust
scala
tsx
None
vue
html
unknown
unknown
unknown
unknown
unknown
unknown
unknown


In [8]:
def parse_ast(file_path: Path):
    file_alias = extension_to_language_name(file_path)
    if not file_alias:
        return None

    lang_name = file_alias
    if not lang_name:
        return None

    try:
        parser = get_parser(lang_name)
        language = get_language(lang_name)
        print(file_alias)
        print(language)


        source_code = file_path.read_text(encoding="utf-8")
        tree = parser.parse(bytes(source_code, "utf-8"))

        return tree
    except Exception:
        print(f"Skipping {file_path.name}: No parser found for language '{lang_name}'")
        return None

In [9]:
for file in files:
    print(parse_ast(Path(file)))

cpp
<Language id=131425438952672, version=15, name="cpp">
cpp
<Language id=131425438952672, version=15, name="cpp">
csharp
<Language id=131425432923360, version=15, name="c_sharp">
go
<Language id=131426052457824, version=14, name=None>
java
<Language id=131425866465792, version=14, name=None>
javascript
<Language id=131425866046688, version=14, name=None>
markdown
<Language id=131425445596384, version=14, name=None>
python
<Language id=131425426730208, version=14, name=None>
rust
<Language id=131425425906912, version=14, name=None>
scala
<Language id=131425423822048, version=15, name="scala">
tsx
<Language id=131425418341664, version=14, name=None>
Skipping palindrome.txt: No parser found for language 'text'
None
vue
<Language id=131425424432352, version=14, name=None>
html
<Language id=131425416523040, version=14, name=None>
Skipping test.csv: No parser found for language 'unknown'
None
Skipping test.doc: No parser found for language 'unknown'
None
Skipping test.docx: No parser found

In [10]:
def print_ast(node, source_bytes: bytes, indent: str = "", is_last: bool = True):
    if node is None:
        return

    marker = "└── " if is_last else "├── "

    # Extract source code string for leaf nodes (nodes with no children)
    snippet = ""
    if len(node.children) == 0:
        token_text = source_bytes[node.start_byte : node.end_byte].decode(
            "utf-8", errors="replace"
        )
        snippet = f" ➔ {token_text!r}"

    # Format line and column coordinates
    pos = f"[{node.start_point[0] + 1}:{node.start_point[1]}]"

    print(f"{indent}{marker}{node.type} {pos}{snippet}")

    # Prepare indentation string for children
    new_indent = indent + ("    " if is_last else "│   ")

    # Recursively traverse child nodes
    count = len(node.children)
    for i, child in enumerate(node.children):
        print_ast(child, source_bytes, new_indent, is_last=(i == count - 1))

In [11]:
file_path = Path(
    "/home/user/Documents/Project_5/backend/experiments/data/code/oops.cpp"
)
tree = parse_ast(file_path)

cpp
<Language id=131425438952672, version=15, name="cpp">


In [12]:
if tree:
    print(tree.root_node)

print()
print()

if tree:
    source_byte = file_path.read_bytes()
    print_ast(tree.root_node, source_byte)

(translation_unit (preproc_include path: (system_lib_string)) (preproc_include path: (system_lib_string)) (preproc_include path: (system_lib_string)) (using_declaration (identifier)) (comment) (comment) (comment) (comment) (class_specifier name: (type_identifier) body: (field_declaration_list (access_specifier) (comment) (field_declaration type: (type_identifier) declarator: (field_identifier)) (field_declaration type: (type_identifier) declarator: (field_identifier)) (field_declaration type: (primitive_type) declarator: (field_identifier)) (access_specifier) (comment) (function_definition type: (primitive_type) declarator: (function_declarator declarator: (field_identifier) parameters: (parameter_list (parameter_declaration type: (primitive_type) declarator: (identifier)))) body: (compound_statement (expression_statement (assignment_expression left: (identifier) right: (identifier))))) (access_specifier) (comment) (function_definition declarator: (function_declarator declarator: (iden

In [13]:
def extract(node, source_bytes):
    text = source_bytes[node.start_byte : node.end_byte].decode("utf-8")
    return text


source_bytes = file_path.read_bytes()

if tree:
    for child in tree.root_node.children:
        raw = extract(child, source_bytes)

In [14]:
file_path = Path(
    "/home/user/Documents/Project_5/backend/experiments/data/code/bank_py/main.py"
)
source_bytes = file_path.read_bytes()
tree = parse_ast(file_path)

if tree:
    print(tree.root_node)
    for child in tree.root_node.children:
        raw = extract(child, source_bytes)

python
<Language id=131425426730208, version=14, name=None>
(module (import_from_statement module_name: (dotted_name (identifier)) name: (dotted_name (identifier)) name: (dotted_name (identifier)) name: (dotted_name (identifier))) (import_from_statement module_name: (dotted_name (identifier)) name: (aliased_import name: (dotted_name (identifier)) alias: (identifier))) (import_from_statement module_name: (dotted_name (identifier)) name: (dotted_name (identifier))) (import_statement name: (aliased_import name: (dotted_name (identifier)) alias: (identifier)) name: (dotted_name (identifier))) (import_statement name: (dotted_name (identifier))) (import_statement name: (dotted_name (identifier))) (import_from_statement module_name: (dotted_name (identifier)) name: (dotted_name (identifier))) (function_definition name: (identifier) parameters: (parameters) body: (block (call function: (identifier) arguments: (argument_list (string (string_start) (string_content (escape_sequence)) (string_end)

#### Parse the AST to make the structured response

In [15]:
CAPTURE_TO_SYMBOL_TYPE = {
    "definition.class": "class",
    "definition.function": "function",
    "definition.method": "method",
    "definition.interface": "interface",
    "definition.type": "enum",  # Enums/Types ko enum/variable me map kar sakte hain
    "definition.macro": "function",
    "definition.variable": "variable",
    "definition.constant": "variable",
}

In [16]:
QUERIES_DIR = Path("/home/user/Documents/Project_5/backend/experiments/data/queries")


def find_parent(node: Node) -> str | None:
    curr = node.parent
    while curr:
        # Har language ke main scope nodes
        if curr.type in [
            "function_definition",  # Python
            "class_definition",  # Python
            "function_declaration",  # JS/TS/Go
            "class_declaration",  # JS/TS/Java
            "method_declaration",  # Java/Go
            "method_definition",  # JS/TS
        ]:
            name_node = curr.child_by_field_name("name")
            if name_node and name_node.text:
                return name_node.text.decode("utf-8")
        curr = curr.parent
    return None


def process_ast(tree: Tree, language: str):
    scm_path: Path = QUERIES_DIR / language / "tags.scm"
    if not scm_path.exists():
        return None

    symbols = []
    relationships = []

    scm_content = scm_path.read_text(encoding="utf-8")
    lang_obj = get_language(language)
    query = Query(lang_obj, scm_content)
    root_node = tree.root_node

    cursor = QueryCursor(query)
    matches = cursor.matches(root_node)

    seen_symbols = set()
    seen_relationships = set()

    for pattern_idx, capture_dict in matches:
        symbol_type = None
        primary_node = None
        symbol_name = None

        if "name" in capture_dict:
            symbol_name = capture_dict["name"][0].text.decode("utf-8")

        # 1. Extract Symbols
        for cap_name, nodes in capture_dict.items():
            if cap_name in CAPTURE_TO_SYMBOL_TYPE:
                symbol_type = CAPTURE_TO_SYMBOL_TYPE[cap_name]
                primary_node = nodes[0]
                break

        if symbol_type and primary_node and symbol_name:
            start_line = primary_node.start_point[0] + 1
            end_line = primary_node.end_point[0] + 1
            parent_name = find_parent(primary_node)

            symbol_key = (symbol_name, symbol_type, start_line)

            if symbol_key not in seen_symbols:
                seen_symbols.add(symbol_key)
                symbol_entry = {
                    "name": symbol_name,
                    "type": symbol_type,
                    "startLine": start_line,
                    "endLine": end_line,
                }
                if parent_name:
                    symbol_entry["parent"] = parent_name

                symbols.append(symbol_entry)

                if parent_name:
                    rel_key = (parent_name, symbol_name, "contains")
                    if rel_key not in seen_relationships:
                        seen_relationships.add(rel_key)
                        relationships.append(
                            {
                                "source": parent_name,
                                "target": symbol_name,
                                "type": "contains",
                            }
                        )

        # 2. Function & Method Calls
        if "reference.call" in capture_dict:
            call_node = capture_dict["reference.call"][0]

            # Agar 'new' expression hai toh clean Class Name extract karein
            if call_node.type == "object_creation_expression":
                type_node = call_node.child_by_field_name("type")
                target_func = (
                    type_node.text.decode("utf-8") if type_node else "anonymous"
                )
            else:
                target_func = symbol_name or (
                    call_node.text.decode("utf-8") if call_node.text else "anonymous"
                )

            caller = find_parent(call_node) or "global"

            rel_key = (caller, target_func, "calls")
            if rel_key not in seen_relationships:
                seen_relationships.add(rel_key)
                relationships.append(
                    {"source": caller, "target": target_func, "type": "calls"}
                )

        # 3. Class Inheritance (Clean Superclass Handling)
        if (
            "reference.class" in capture_dict
            or "reference.implementation" in capture_dict
        ):
            ref_nodes = capture_dict.get("reference.class") or capture_dict.get(
                "reference.implementation"
            )
            if ref_nodes:
                ref_node = ref_nodes[0]

                # FIX 1 & 2: Object Creation ('new') ko inherits me na jane dein
                if ref_node.type == "object_creation_expression":
                    continue

                # FIX 1: Superclass node me se sirf class identifier nikalen ('extends BankAccount' -> 'BankAccount')
                if ref_node.type == "superclass":
                    type_child = (
                        ref_node.child_by_field_name("type") or ref_node.children[-1]
                    )
                    target_class = (
                        type_child.text.decode("utf-8")
                        if type_child
                        else ref_node.text.decode("utf-8")
                    )
                else:
                    target_class = (
                        ref_node.text.decode("utf-8") if ref_node.text else "anonymous"
                    )

                source_class = find_parent(ref_node)
                rel_type = (
                    "inherits" if "reference.class" in capture_dict else "implements"
                )

                if source_class and target_class:
                    # Clean any trailing/leading keywords just in case
                    target_class = (
                        target_class.replace("extends", "")
                        .replace("implements", "")
                        .strip()
                    )

                    rel_key = (source_class, target_class, rel_type)
                    if rel_key not in seen_relationships:
                        seen_relationships.add(rel_key)
                        relationships.append(
                            {
                                "source": source_class,
                                "target": target_class,
                                "type": rel_type,
                            }
                        )

    return symbols, relationships

In [17]:
def parse_ast_and_print_structure(file_path: Path):
    file_alias = extension_to_language_name(file_path)
    print(file_alias)
    if not file_alias:
        return None

    lang_name = file_alias
    if not lang_name:
        return None

    try:
        parser = get_parser(lang_name)
        language = get_language(lang_name)
        # print(language.)

        source_code = file_path.read_text(encoding="utf-8")
        tree = parser.parse(bytes(source_code, "utf-8"))

        return process_ast(tree, file_alias)
    except Exception:
        print(f"Skipping {file_path.name}: No parser found for language '{lang_name}'")
        return None

In [18]:
file_path = Path(
    "/home/user/Documents/Project_5/backend/experiments/data/code/bank.java"
)
print(parse_ast_and_print_structure(file_path))

java
None


In [19]:
# one small table per language — just node type names, nothing fancy
CHUNK_NODE_TYPES = {
    "python": {"function_definition", "class_definition"},
    "javascript": {"function_declaration", "class_declaration", "method_definition"},
    "go": {"function_declaration", "method_declaration", "type_declaration"},
    "java": {"method_declaration", "class_declaration", "interface_declaration"},
    "rust": {"function_item", "impl_item", "struct_item"},
}


def extract_chunks(node, source_bytes, lang, chunks=None):
    # print(node)
    if chunks is None:
        chunks = []
    if node.type in CHUNK_NODE_TYPES[lang]:
        chunks.append(
            {
                "type": node.type,
                "start_byte": node.start_byte,
                "end_byte": node.end_byte,
                "start_line": node.start_point[0],
                "content": source_bytes[node.start_byte : node.end_byte].decode(
                    "utf-8"
                ),
            }
        )
        return chunks  # don't recurse into it — whole node is one chunk
    for child in node.children:
        extract_chunks(child, source_bytes, lang, chunks)
    return chunks

In [20]:
def parse_ast_and_print_structure(file_path: Path):
    file_alias = extension_to_language_name(file_path)
    print(file_alias)
    if not file_alias:
        return None

    lang_name = file_alias
    if not lang_name:
        return None

    try:
        parser = get_parser(lang_name)
        language = get_language(lang_name)
        # print(language.)

        source_code = file_path.read_text(encoding="utf-8")
        tree = parser.parse(bytes(source_code, "utf-8"))
        print(tree.root_node)
        return extract_chunks(tree.root_node, bytes(source_code, "utf-8"), lang_name)
    except Exception:
        print(f"Skipping {file_path.name}: No parser found for language '{lang_name}'")
        return None

In [21]:
file_path = Path("/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_java/src/com/example/vehicle/Main.java")
source_bytes = file_path.read_bytes()
tree = parse_ast(file_path)
print(print_ast(tree.root_node, source_bytes))

java
<Language id=131425866465792, version=14, name=None>
└── program [1:0]
    ├── line_comment [1:0] ➔ '// src/vehicle/Main.java'
    ├── package_declaration [2:0]
    │   ├── package [2:0] ➔ 'package'
    │   ├── scoped_identifier [2:8]
    │   │   ├── scoped_identifier [2:8]
    │   │   │   ├── identifier [2:8] ➔ 'com'
    │   │   │   ├── . [2:11] ➔ '.'
    │   │   │   └── identifier [2:12] ➔ 'example'
    │   │   ├── . [2:19] ➔ '.'
    │   │   └── identifier [2:20] ➔ 'app'
    │   └── ; [2:23] ➔ ';'
    ├── import_declaration [4:0]
    │   ├── import [4:0] ➔ 'import'
    │   ├── scoped_identifier [4:7]
    │   │   ├── scoped_identifier [4:7]
    │   │   │   ├── identifier [4:7] ➔ 'com'
    │   │   │   ├── . [4:10] ➔ '.'
    │   │   │   └── identifier [4:11] ➔ 'example'
    │   │   ├── . [4:18] ➔ '.'
    │   │   └── identifier [4:19] ➔ 'vehicle'
    │   ├── . [4:26] ➔ '.'
    │   ├── asterisk [4:27]
    │   │   └── * [4:27] ➔ '*'
    │   └── ; [4:28] ➔ ';'
    ├── import_declaratio

In [22]:
# chunker.py
# Hierarchical AST-based chunk extractor with comment attachment,
# type/interface support, and module-level variable filtering.

CHUNK_NODE_TYPES = {
    "python": {
        "class_definition": "class",
        "function_definition": "function",
    },
    "javascript": {
        "class_declaration": "class",
        "function_declaration": "function",
        "method_definition": "function",
        # arrow/function-expression consts handled separately (pattern match, not type match)
        "interface_declaration": "interface",  # only present when parsing .ts/.tsx grammar
        "type_alias_declaration": "type",
    },
    "typescript": {
        "class_declaration": "class",
        "function_declaration": "function",
        "method_definition": "function",
        "interface_declaration": "interface",
        "type_alias_declaration": "type",
        "enum_declaration": "enum",
    },
    "go": {
        "function_declaration": "function",
        "method_declaration": "function",
        "type_declaration": "type",  # covers struct + interface, disambiguated below
    },
    "java": {
        "class_declaration": "class",
        "interface_declaration": "interface",
        "method_declaration": "function",
        "enum_declaration": "enum",
    },
    "rust": {
        "function_item": "function",
        "impl_item": "class",  # closest analogue: groups methods for a type
        "struct_item": "type",
        "enum_item": "enum",
        "trait_item": "interface",
    },
}

# Node types that ARE function/method-like, used to decide whether to
# recurse into a class body for per-method chunks.
METHOD_KINDS = {"function"}

# Minimum size (in lines) for a NESTED function to get its own chunk.
# Below this, it stays inline as part of the parent chunk's text.
NESTED_FN_LINE_THRESHOLD = 10


# Only pull OUT module-level variables that look meaningful (exported,
# UPPER_CASE constants, or config-like objects) -- not every `let i = 0`.
def _is_significant_variable(name: str | None, value_node, include_all_globals) -> bool:
    print(include_all_globals)
    """Check if variable should be extracted. Set include_all_globals=True to retain all global vars."""
    if not name:
        return False
    if include_all_globals:
        print(f"include_all_globals is True, including variable: {name}")
        return True
    if name.isupper():  # CONSTANT_CASE
        return True
    if value_node and value_node.type in (
        "object",
        "object_pattern",
        "dictionary",
        "call_expression",
        "array",
    ):
        return True
    return False


def _text(node, source_bytes) -> str:
    return source_bytes[node.start_byte : node.end_byte].decode("utf-8")


def _leading_comment(node, source_bytes):
    """Walk backwards through preceding siblings, collecting a contiguous
    comment block immediately above this node (no blank-line gap)."""
    prev = node.prev_sibling
    # print("Prev: ", prev)
    collected = []
    expected_end_row = node.start_point[0]  # comment must end right before this row
    # print(prev)
    i = 0
    # print(prev.type)
    # print(prev is not None and prev.type == "comment")

    while prev is not None and (
        prev.type == "line_comment"
        or prev.type == "block_comment"
        or prev.type == "comment"
    ):
        # print(f"iteration {i}")
        # gap = expected_end_row - prev.end_point[0]
        # if gap > 1:  # blank line between comment and code -> not attached
        #     break
        collected.insert(0, _text(prev, source_bytes))
        expected_end_row = prev.start_point[0]
        prev = prev.prev_sibling
        i += 1
    comments = "\n".join(collected) if collected else None
    # print(comments)
    return comments


def _kind_of(node_type: str, lang: str, node=None):
    # print("Lang:    ", lang)
    kind = CHUNK_NODE_TYPES[lang].get(node_type)
    # print("Kind:    ", kind)
    # Go's type_declaration covers both struct and interface -- disambiguate
    if lang == "go" and node_type == "type_declaration" and node is not None:
        text = node.type
        # crude check on child type_spec's underlying type node
        for child in node.children:
            if child.type == "type_spec":
                for gc in child.children:
                    if gc.type == "interface_type":
                        return "interface"
                    if gc.type == "struct_type":
                        return "type"
    return kind


def _add_to_global_chunk(comment_text, code_text, chunks, node, lang):
    if not chunks or chunks[0].get("kind") != "global_variables":
        return

    parts = [chunks[0]["content"]]
    if comment_text:
        parts.append(comment_text)
    parts.append(code_text)
    chunks[0]["content"] = "\n".join(filter(None, parts)).strip()

    node_type = node.type
    kind = _kind_of(node_type, lang, node)


def extract_chunks(
    node,
    source_bytes,
    lang,
    include_all_globals=False,
    merge_all_global_var=False,
):
    chunks = []

    # Agar global vars merge karne hain, to shuru me hi 1st chunk bana do
    if merge_all_global_var:
        print(merge_all_global_var)
        chunks.append(
            {
                "kind": "global_variables",
                "name": "Global Scope Variables",
                "content": "",
                "comment": None,
                "start_line": -1,
                "end_line": -1,
            }
        )

    # Ab actual recursion start karo
    _extract_chunks_rec(
        node,
        source_bytes,
        lang,
        chunks=chunks,
        include_all_globals=include_all_globals,
        merge_all_global_var=merge_all_global_var,
    )

    # # Clean up: Agar koi global variable nahi mila, to khali chunk hata do
    # if merge_all_global_var and chunks and chunks[0]["kind"] == "global_variables":
    #     if not chunks[0]["content"].strip():
    #         chunks.pop(0)

    return chunks


def _extract_chunks_rec(
    node: Node,
    source_bytes,
    lang,
    chunks: list[dict],
    parent_class=None,
    parent_function=None,
    depth=0,
    include_all_globals=False,
    merge_all_global_var=False,
):
    if chunks is None:
        # print("Chunk does")
        return None

    node_type = node.type
    if node.type != "program":
        # print(node.type)
        print(f"Node: {node.type} Name: {node.text.decode("utf-8")}")
    kind = _kind_of(node_type, lang, node)

    # -------------------------------------------------------------
    # 1. JS/TS Lexical & Variable Declarations (const / let / var)
    # -------------------------------------------------------------
    if lang in ("javascript", "typescript") and node_type in (
        "lexical_declaration",
        "variable_declaration",
    ):
        comment = _leading_comment(node, source_bytes)
        for child in node.children:
            if child.type == "variable_declarator":
                name_node = child.child_by_field_name("name")
                value_node = child.child_by_field_name("value")

                var_name = _text(name_node, source_bytes) if name_node else None

                # Check if value is an Arrow Function or Function Expression
                if value_node and value_node.type in (
                    "arrow_function",
                    "function_expression",
                    "generator_function",
                ):
                    fn_name = var_name or "<anonymous>"
                    n_lines = node.end_point[0] - node.start_point[0]

                    if (
                        parent_function is not None
                        and n_lines < NESTED_FN_LINE_THRESHOLD
                    ):
                        continue

                    chunks.append(
                        {
                            "kind": "function",
                            "name": fn_name,
                            "is_arrow": value_node.type == "arrow_function",
                            "comment": comment,
                            "parent_class": parent_class,
                            "parent_function": parent_function,
                            "content": _text(node, source_bytes),
                            "start_line": node.start_point[0],
                            "end_line": node.end_point[0],
                        }
                    )
                # Otherwise, treat as Module/Global Level Variable
                elif parent_class is None and parent_function is None:
                    print(
                        f"Checking variable: {var_name}, include_all_globals={include_all_globals}, merge_all_global_var={merge_all_global_var}"
                    )
                    if _is_significant_variable(
                        var_name, value_node, include_all_globals
                    ):
                        content = _text(node, source_bytes)
                        if merge_all_global_var:
                            _add_to_global_chunk(comment, content, chunks, node, lang)
                        else:
                            chunks.append(
                                {
                                    "kind": "variable",
                                    "name": var_name,
                                    "comment": comment,
                                    "content": content,
                                    "start_line": node.start_point[0],
                                    "end_line": node.end_point[0],
                                }
                            )
        return chunks

    # -------------------------------------------------------------
    # 2. Classes, Interfaces, Types, Enums
    # -------------------------------------------------------------
    if kind in ("class", "interface", "type", "enum"):
        name_node = node.child_by_field_name("name")
        # print("Name Node:", name_node)
        name = _text(name_node, source_bytes) if name_node else None
        comment = _leading_comment(node, source_bytes)
        # print(comment)

        if kind == "class":
            # 1) class-summary chunk: signature-only, not full bodies
            method_sigs = []
            body = node.child_by_field_name("body")
            if body:
                for child in body.children:
                    if _kind_of(child.type, lang, child) == "function":
                        mname_node = child.child_by_field_name("name")
                        mname = _text(mname_node, source_bytes) if mname_node else "?"
                        method_sigs.append(mname)
            chunks.append(
                {
                    "kind": "class_summary",
                    "name": name,
                    "comment": comment,
                    "content": f"class {name}: methods = {', '.join(method_sigs)}",
                    "start_line": node.start_point[0],
                    "end_line": node.end_point[0],
                }
            )
            # 2) recurse so each method becomes its own chunk, tagged with parent_class
            if body:
                for child in body.children:
                    _extract_chunks_rec(
                        child,
                        source_bytes,
                        lang,
                        parent_class=name,
                        depth=depth + 1,
                        chunks=chunks,
                        merge_all_global_var=merge_all_global_var,
                        include_all_globals=include_all_globals,
                    )
            return chunks
        else:
            # interface / type / enum -> single chunk, no further recursion needed
            chunks.append(
                {
                    "kind": kind,
                    "name": name,
                    "comment": comment,
                    "content": _text(node, source_bytes),
                    "start_line": node.start_point[0],
                    "end_line": node.end_point[0],
                }
            )
            return chunks

    # -------------------------------------------------------------
    # 3. Functions
    # -------------------------------------------------------------
    if kind == "function":
        name_node = node.child_by_field_name("name")
        # if node.type == "lexical_declaration":
        #     print(node)
        #     print(_text(name_node, source_bytes) if name_node else None)
        #     print("Name Node: ", name_node)
        name = _text(name_node, source_bytes) if name_node else None
        comment = _leading_comment(node, source_bytes)
        n_lines = node.end_point[0] - node.start_point[0]

        # nested + small -> don't split out, let it stay inside parent's text
        if parent_function is not None and n_lines < NESTED_FN_LINE_THRESHOLD:
            return chunks  # caller already captured full parent text

        chunks.append(
            {
                "kind": "function",
                "name": name,
                "comment": comment,
                "parent_class": parent_class,
                "parent_function": parent_function,
                "content": _text(node, source_bytes),
                "start_line": node.start_point[0],
                "end_line": node.end_point[0],
            }
        )
        # still recurse, in case there's a sizeable nested function inside
        for child in node.children:
            _extract_chunks_rec(
                child,
                source_bytes,
                lang,
                parent_class=parent_class,
                parent_function=name,
                depth=depth + 1,
                chunks=chunks,
                merge_all_global_var=merge_all_global_var,
                include_all_globals=include_all_globals,
            )
        return chunks

    # module-level variable/const, only if it looks significant, only at depth 0
    if depth == 0 and node_type in ("assignment", "variable_declarator", "const_spec"):
        name_node = node.child_by_field_name("name") or node.child_by_field_name("left")
        value_node = node.child_by_field_name("value") or node.child_by_field_name(
            "right"
        )
        name = _text(name_node, source_bytes) if name_node else None
        if _is_significant_variable(name, value_node, include_all_globals):
            comment = _leading_comment(node, source_bytes)
            content = _text(node, source_bytes)
            if merge_all_global_var:
                _add_to_global_chunk(comment, content, chunks, node, lang)
            else:
                chunks.append(
                    {
                        "kind": "variable",
                        "name": name,
                        "comment": _leading_comment(node, source_bytes),
                        "content": _text(node, source_bytes),
                        "start_line": node.start_point[0],
                        "end_line": node.end_point[0],
                    }
                )
        return chunks  # don't recurse into it either way

    # -------------------------------------------------------------
    # 4. Python & General Module-Level Variables / Assignments
    # -------------------------------------------------------------
    if (
        parent_class is None
        and parent_function is None
        and node_type in ("assignment", "const_spec", "expression_statement", "variable_declarator")
    ):
        target_node = node
        if node_type == "expression_statement" and len(node.children) > 0:
            target_node = node.children[0]

        if target_node.type in ("assignment", "augmented_assignment"):
            name_node = target_node.child_by_field_name(
                "left"
            ) or target_node.child_by_field_name("name")
            value_node = target_node.child_by_field_name(
                "right"
            ) or target_node.child_by_field_name("value")

            var_name = _text(name_node, source_bytes) if name_node else None

            if _is_significant_variable(var_name, value_node, include_all_globals):
                comment = _leading_comment(node, source_bytes)
                content = _text(node, source_bytes)
                if merge_all_global_var:
                    _add_to_global_chunk(comment, content, chunks, node, lang)
                else:
                    chunks.append(
                        {
                            "kind": "variable",
                            "name": var_name,
                            "comment": comment,
                            "content": content,
                            "start_line": node.start_point[0],
                            "end_line": node.end_point[0],
                        }
                    )
            return chunks

    # default: keep walking
    for child in node.children:
        _extract_chunks_rec(
            child,
            source_bytes,
            lang,
            parent_class=parent_class,
            parent_function=parent_function,
            depth=depth,
            chunks=chunks,
            include_all_globals=include_all_globals,
            merge_all_global_var=merge_all_global_var,
        )
    return chunks

In [23]:
file_path = Path("/home/user/Documents/Project_5/backend/experiments/data/code/bank_py/main.py")
tree = parse_ast(file_path)
lang_name = extension_to_language_name(file_path)
source_bytes = file_path.read_bytes()

if tree:
    print(extract_chunks(tree.root_node, source_bytes, lang_name, include_all_globals=True, merge_all_global_var=True))

python
<Language id=131425426730208, version=14, name=None>
True
Node: module Name: from savings_account import SavingsAccount, sayWhat, sayWhy
from checking_account import CheckingAccount as cb
from os import mkdir
import math as m, os
import bank_account
import http
from fastapi import FastAPI


def main():
    print("=== Object-Oriented Programming (OOP) in Python Demo ===\n")

    # Creating objects (Python handles polymorphism natively without pointers)
    acc1 = SavingsAccount("SA101", "Alice Smith", 1000.0, 4.5)
    acc2 = cb("CA201", "Bob Jones", 500.0, 200.0)

    print("--- 1. Savings Account Operations ---")
    print(f"Holder: {acc1.get_account_holder()} | Initial Balance: ${acc1.get_balance():.2f}")
    acc1.deposit(200.0)
    acc1.calculate_interest()  # Polymorphic call
    acc1.withdraw(1200.0)      # Should fail due to $50 minimum balance rule
    acc1.withdraw(100.0)       # Should succeed

    print("\n--- 2. Checking Account Operations ---")
    print(f"Holder: {ac

In [24]:
[
    {
        "kind": "global_variables",
        "name": "Global Scope Variables",
        "content": 'num = 0\nname = "Hello"\nfirst = "A"',
        "comment": None,
        "start_line": -1,
        "end_line": -1,
    },
    {
        "kind": "class_summary",
        "name": "BankAccount",
        "comment": "// ==========================================\n// 1. ABSTRACTION & ENCAPSULATION\n// ==========================================\n// Abstract class providing a blueprint for all bank accounts",
        "content": "class BankAccount: methods = updateBalance, getAccountNumber, getAccountHolder, getBalance, calculateInterest, deposit, withdraw",
        "start_line": 9,
        "end_line": 61,
    },
    {
        "kind": "function",
        "name": "updateBalance",
        "comment": "// Protected members are accessible by derived classes",
        "parent_class": "BankAccount",
        "parent_function": None,
        "content": "protected void updateBalance(double amount) {\n        this.balance += amount;\n    }",
        "start_line": 23,
        "end_line": 25,
    },
    {
        "kind": "function",
        "name": "getAccountNumber",
        "comment": "// Getter methods (Encapsulation - controlled access)",
        "parent_class": "BankAccount",
        "parent_function": None,
        "content": "public String getAccountNumber() {\n        return accountNumber;\n    }",
        "start_line": 28,
        "end_line": 30,
    },
    {
        "kind": "function",
        "name": "getAccountHolder",
        "comment": None,
        "parent_class": "BankAccount",
        "parent_function": None,
        "content": "public String getAccountHolder() {\n        return accountHolder;\n    }",
        "start_line": 32,
        "end_line": 34,
    },
    {
        "kind": "function",
        "name": "getBalance",
        "comment": None,
        "parent_class": "BankAccount",
        "parent_function": None,
        "content": "public double getBalance() {\n        return balance;\n    }",
        "start_line": 36,
        "end_line": 38,
    },
    {
        "kind": "function",
        "name": "calculateInterest",
        "comment": "// Pure Virtual Function equivalent (Abstract method)",
        "parent_class": "BankAccount",
        "parent_function": None,
        "content": "public abstract void calculateInterest();",
        "start_line": 41,
        "end_line": 41,
    },
    {
        "kind": "function",
        "name": "deposit",
        "comment": "// Common method",
        "parent_class": "BankAccount",
        "parent_function": None,
        "content": 'public void deposit(double amount) {\n        if (amount > 0) {\n            this.balance += amount;\n            System.out.println("Deposited: $" + amount + " | New Balance: $" + this.balance);\n        } else {\n            System.out.println("Invalid deposit amount!");\n        }\n    }',
        "start_line": 44,
        "end_line": 51,
    },
    {
        "kind": "function",
        "name": "withdraw",
        "comment": None,
        "parent_class": "BankAccount",
        "parent_function": None,
        "content": 'public void withdraw(double amount) {\n        if (amount > 0 && amount <= this.balance) {\n            this.balance -= amount;\n            System.out.println("Withdrew: $" + amount + " | Remaining Balance: $" + this.balance);\n        } else {\n            System.out.println("Invalid withdrawal amount or insufficient funds!");\n        }\n    }',
        "start_line": 53,
        "end_line": 60,
    },
    {
        "kind": "class_summary",
        "name": "SavingsAccount",
        "comment": "// ==========================================\n// 2. INHERITANCE & POLYMORPHISM\n// ==========================================\n// SavingsAccount inherits from BankAccount",
        "content": "class SavingsAccount: methods = calculateInterest, withdraw",
        "start_line": 67,
        "end_line": 93,
    },
    {
        "kind": "function",
        "name": "calculateInterest",
        "comment": "// Overriding the abstract method (Polymorphism)",
        "parent_class": "SavingsAccount",
        "parent_function": None,
        "content": '@Override\n    public void calculateInterest() {\n        double interest = getBalance() * (interestRate / 100);\n        deposit(interest);\n        System.out.println("Interest added: $" + interest + " (Rate: " + interestRate + "%)");\n    }',
        "start_line": 76,
        "end_line": 81,
    },
    {
        "kind": "function",
        "name": "withdraw",
        "comment": "// Overriding the withdraw method (Runtime Polymorphism / Method Overriding)",
        "parent_class": "SavingsAccount",
        "parent_function": None,
        "content": '@Override\n    public void withdraw(double amount) {\n        // Savings account rule: Cannot withdraw if balance drops below $50\n        if (getBalance() - amount < 50.0) {\n            System.out.println("Withdrawal denied! Savings account must maintain a minimum balance of $50.");\n        } else {\n            super.withdraw(amount);\n        }\n    }',
        "start_line": 84,
        "end_line": 92,
    },
    {
        "kind": "class_summary",
        "name": "CheckingAccount",
        "comment": "// Another derived class: CheckingAccount",
        "content": "class CheckingAccount: methods = calculateInterest, withdraw",
        "start_line": 96,
        "end_line": 123,
    },
    {
        "kind": "function",
        "name": "calculateInterest",
        "comment": None,
        "parent_class": "CheckingAccount",
        "parent_function": None,
        "content": '@Override\n    public void calculateInterest() {\n        System.out.println("Checking accounts do not earn interest.");\n    }',
        "start_line": 104,
        "end_line": 107,
    },
    {
        "kind": "function",
        "name": "withdraw",
        "comment": "// Overriding withdraw to allow overdraft",
        "parent_class": "CheckingAccount",
        "parent_function": None,
        "content": '@Override\n    public void withdraw(double amount) {\n        if (amount > 0 && amount <= (getBalance() + overdraftLimit)) {\n            double newBalance = getBalance() - amount;\n            System.out.println("Withdrew: $" + amount + " using overdraft protection.");\n            if (newBalance < 0) {\n                System.out.println("Warning: Account is in overdraft!");\n            }\n            super.withdraw(amount);\n        } else {\n            System.out.println("Withdrawal exceeds overdraft limit!");\n        }\n    }',
        "start_line": 110,
        "end_line": 122,
    },
    {
        "kind": "class_summary",
        "name": "Main",
        "comment": "// ==========================================\n// MAIN CLASS (Demonstration)\n// ==========================================\n/* \n    THIS METHOD CALLS EACH AND EVERY METHOD\n    CREATES NEW INSTANCE OF EACH OF THE CLASS\n*/",
        "content": "class Main: methods = main",
        "start_line": 132,
        "end_line": 152,
    },
    {
        "kind": "function",
        "name": "main",
        "comment": None,
        "parent_class": "Main",
        "parent_function": None,
        "content": 'public static void main(String[] args) {\n        System.out.println("=== Object-Oriented Programming (OOP) in Java Demo ===\\n");\n\n        // Creating objects using polymorphism (Base class references)\n        BankAccount acc1 = new SavingsAccount("SA101", "Alice Smith", 1000.0, 4.5);\n        BankAccount acc2 = new CheckingAccount("CA201", "Bob Jones", 500.0, 200.0);\n\n        System.out.println("--- 1. Savings Account Operations ---");\n        System.out.println("Holder: " + acc1.getAccountHolder() + " | Initial Balance: $" + acc1.getBalance());\n        acc1.deposit(200.0);\n        acc1.calculateInterest(); // Polymorphic call\n        acc1.withdraw(1200.0);    // Should fail due to $50 minimum balance rule\n        acc1.withdraw(100.0);     // Should succeed\n\n        System.out.println("\\n--- 2. Checking Account Operations ---");\n        System.out.println("Holder: " + acc2.getAccountHolder() + " | Initial Balance: $" + acc2.getBalance());\n        acc2.withdraw(600.0);     // Uses overdraft protection\n        acc2.calculateInterest(); // Polymorphic call\n    }',
        "start_line": 133,
        "end_line": 151,
    },
]

[{'kind': 'global_variables',
  'name': 'Global Scope Variables',
  'content': 'num = 0\nname = "Hello"\nfirst = "A"',
  'comment': None,
  'start_line': -1,
  'end_line': -1},
 {'kind': 'class_summary',
  'name': 'BankAccount',
  'comment': '// ==========================================\n// 1. ABSTRACTION & ENCAPSULATION\n// ==========================================\n// Abstract class providing a blueprint for all bank accounts',
  'content': 'class BankAccount: methods = updateBalance, getAccountNumber, getAccountHolder, getBalance, calculateInterest, deposit, withdraw',
  'start_line': 9,
  'end_line': 61},
 {'kind': 'function',
  'name': 'updateBalance',
  'comment': '// Protected members are accessible by derived classes',
  'parent_class': 'BankAccount',
  'parent_function': None,
  'content': 'protected void updateBalance(double amount) {\n        this.balance += amount;\n    }',
  'start_line': 23,
  'end_line': 25},
 {'kind': 'function',
  'name': 'getAccountNumber',
  'comme

In [25]:
import os
from pathlib import Path

# ==============================================================================
# 1. LANGUAGE IMPORT CONFIGURATION DICTIONARY
# ==============================================================================
LANG_IMPORT_CONFIG = {
    "javascript": {
        "import_nodes": ["import_statement"],
        "require_nodes": ["call_expression"],
    },
    "typescript": {
        "import_nodes": ["import_statement"],
        "require_nodes": ["call_expression"],
    },
    "python": {
        "import_nodes": ["import_statement", "import_from_statement"],
    },
    "java": {
        "import_nodes": ["import_declaration"],
    },
    "go": {
        "import_nodes": ["import_declaration", "import_spec"],
    },
    "rust": {
        "import_nodes": ["use_declaration"],
    },
}

# Supported file extensions for resolving local relative paths
FILE_EXTENSIONS = {
    "python": [".py"],
    "javascript": [".js", ".jsx", ".mjs"],
    "typescript": [".ts", ".tsx"],
    "java": [".java"],
    "go": [".go"],
    "rust": [".rs"],
}


# ==============================================================================
# 2. LOCAL PATH RESOLVER ENGINE
# ==============================================================================
def resolve_local_path(source_file_path: str, import_path: str, repo_root: str, lang: str) -> str:
    """
    Translates relative/local import strings (e.g. './user_active' or 'crate::user')
    into the full relative repository path (e.g. 'src/user/user_active.py').
    """
    source_dir = os.path.dirname(source_file_path)
    possible_paths = []
    exts = FILE_EXTENSIONS.get(lang, [""])

    # --- JS / TS / Python Relative Imports ---
    if import_path.startswith((".", "/")):
        base_target = os.path.normpath(os.path.join(source_dir, import_path))
        
        # 1. Direct file match with extension: e.g., user_active.py
        for ext in exts:
            possible_paths.append(f"{base_target}{ext}")
            
        # 2. Index file in directory: e.g., user_active/index.ts
        for ext in exts:
            possible_paths.append(os.path.join(base_target, f"index{ext}"))
            possible_paths.append(os.path.join(base_target, f"mod{ext}"))

    # --- Rust Crate / Module Imports ---
    elif lang == "rust" and (import_path.startswith("crate::") or import_path.startswith("super::")):
        clean_path = import_path.replace("crate::", "").replace("super::", "").replace("::", "/")
        base_target = os.path.join(repo_root, "src", clean_path)
        possible_paths.append(f"{base_target}.rs")
        possible_paths.append(os.path.join(base_target, "mod.rs"))

    # Check which path actually exists on the disk
    for path in possible_paths:
        if os.path.exists(path):
            # Return relative path from repo root for consistency
            return os.path.relpath(path, repo_root)

    # If file doesn't exist on disk, return best-guess relative path
    return os.path.relpath(possible_paths[0] if possible_paths else import_path, repo_root)


# ==============================================================================
# 3. STRUCTURED RELATIONSHIP EXTRACTOR
# ==============================================================================
def extract_relationships(root_node, source_bytes: bytes, source_file_path: str, repo_root: str, lang: str):
    """
    Extracts import dependencies from AST and returns a list of structured graph edge dictionaries.
    """
    relationships = []
    source_rel_path = os.path.relpath(source_file_path, repo_root)

    def _text(node):
        if not node:
            return ""
        return source_bytes[node.start_byte : node.end_byte].decode("utf-8")

    def create_edge(target, symbols, is_local):
        resolved_target = target
        if is_local:
            resolved_target = resolve_local_path(source_file_path, target, repo_root, lang)

        return {
            "source": source_rel_path,
            "target": resolved_target,
            "imported_symbols": symbols,
            "source_file_path": os.path.abspath(source_file_path),
            "type": "imports",  # Options: "imports" | "inherits" | "implements" | "calls" | "contains"
            "is_external": not is_local
        }

    def walk(node):
        node_type = node.type

        # ----------------------------------------------------------------------
        # A. PYTHON
        # ----------------------------------------------------------------------
        if lang == "python":
            if node_type == "import_statement":
                # import os, sys
                for child in node.children:
                    if child.type == "dotted_name":
                        pkg = _text(child)
                        relationships.append(create_edge(pkg, ["*"], is_local=False))

            elif node_type == "import_from_statement":
                # from .user_active import User, get_status
                module_node = node.child_by_field_name("module_name")
                raw_text = _text(node)
                
                # Extract imported symbol names
                symbols = []
                for child in node.children:
                    if child.type in ("import_list", "aliased_import", "identifier"):
                        if child.type != "import":
                            symbols.append(_text(child))
                
                is_relative = raw_text.startswith("from .") or raw_text.startswith("from ..")
                mod_name = _text(module_node) if module_node else ""

                if is_relative:
                    dots = raw_text.split("import")[0].replace("from", "").strip()
                    target_path = f"{dots}/{mod_name}".replace(".", "/")
                    relationships.append(create_edge(target_path, symbols, is_local=True))
                else:
                    pkg = mod_name.split(".")[0]
                    relationships.append(create_edge(pkg, symbols, is_local=False))

        # ----------------------------------------------------------------------
        # B. JAVASCRIPT / TYPESCRIPT
        # ----------------------------------------------------------------------
        elif lang in ("javascript", "typescript"):
            if node_type == "import_statement":
                source_node = node.child_by_field_name("source")
                if source_node:
                    import_path = _text(source_node).strip("'\"`")
                    is_local = import_path.startswith((".", "/", "~", "@/"))
                    
                    # Extract imported clause/symbols
                    symbols = []
                    clause = node.child_by_field_name("clause")
                    if clause:
                        symbols.append(_text(clause))
                    else:
                        symbols.append("*")

                    relationships.append(create_edge(import_path, symbols, is_local=is_local))

        # ----------------------------------------------------------------------
        # C. JAVA
        # ----------------------------------------------------------------------
        elif lang == "java":
            if node_type == "import_declaration":
                # import com.example.user.UserActive;
                raw_import = _text(node).replace("import", "").replace(";", "").strip()
                symbols = [raw_import.split(".")[-1]]
                
                # Check if it belongs to local project package structure
                is_local = raw_import.startswith("com.myproject")  # Customize project base package
                relationships.append(create_edge(raw_import, symbols, is_local=is_local))

        # ----------------------------------------------------------------------
        # D. GO
        # ----------------------------------------------------------------------
        elif lang == "go":
            if node_type == "import_spec":
                # import "github.com/gin-gonic/gin" OR import "./user"
                path_node = node.child_by_field_name("path")
                if path_node:
                    import_path = _text(path_node).strip('"')
                    is_local = import_path.startswith((".", "/")) or "my_project_name" in import_path
                    relationships.append(create_edge(import_path, ["*"], is_local=is_local))

        # ----------------------------------------------------------------------
        # E. RUST
        # ----------------------------------------------------------------------
        elif lang == "rust":
            if node_type == "use_declaration":
                # use crate::user::user_active::User; OR use std::collections::HashMap;
                raw_use = _text(node).replace("use", "").replace(";", "").strip()
                symbols = [raw_use.split("::")[-1]]
                is_local = raw_use.startswith("crate::") or raw_use.startswith("super::") or raw_use.startswith("self::")
                relationships.append(create_edge(raw_use, symbols, is_local=is_local))

        # Recurse down AST
        for child in node.children:
            walk(child)

    walk(root_node)
    return relationships

In [26]:
path = "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/"
file_path = Path(f"{path}/main.rs")
tree = parse_ast(file_path)
lang_name = extension_to_language_name(file_path)
if tree:
    print(
        extract_relationships(
            tree.root_node,
            file_path.read_bytes(),
            str(file_path),
            path,
            lang_name,
        )
    )

rust
<Language id=131425425906912, version=14, name=None>
[{'source': 'main.rs', 'target': 'vehicle::Vehicle', 'imported_symbols': ['Vehicle'], 'source_file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/main.rs', 'type': 'imports', 'is_external': True}, {'source': 'main.rs', 'target': 'src/car/{*}.rs', 'imported_symbols': ['{*}'], 'source_file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/main.rs', 'type': 'imports', 'is_external': False}, {'source': 'main.rs', 'target': 'electric_car::ElectricCar', 'imported_symbols': ['ElectricCar'], 'source_file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/main.rs', 'type': 'imports', 'is_external': True}]


In [27]:
[
    {
        "source": "main.rs",
        "target": "vehicle::Vehicle",
        "imported_symbols": ["Vehicle"],
        "source_file_path": "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/main.rs",
        "type": "imports",
        "is_external": True,
    },
    {
        "source": "main.rs",
        "target": "car::Car",
        "imported_symbols": ["Car"],
        "source_file_path": "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/main.rs",
        "type": "imports",
        "is_external": True,
    },
    {
        "source": "main.rs",
        "target": "electric_car::ElectricCar",
        "imported_symbols": ["ElectricCar"],
        "source_file_path": "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/main.rs",
        "type": "imports",
        "is_external": True,
    },
]

[{'source': 'main.rs',
  'target': 'vehicle::Vehicle',
  'imported_symbols': ['Vehicle'],
  'source_file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/main.rs',
  'type': 'imports',
  'is_external': True},
 {'source': 'main.rs',
  'target': 'car::Car',
  'imported_symbols': ['Car'],
  'source_file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/main.rs',
  'type': 'imports',
  'is_external': True},
 {'source': 'main.rs',
  'target': 'electric_car::ElectricCar',
  'imported_symbols': ['ElectricCar'],
  'source_file_path': '/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_rust/src/main.rs',
  'type': 'imports',
  'is_external': True}]

In [28]:
import_aliases = {
    "javascript": {
        "import_node": ["import_statement", "lexical_declaration"],
        "get_imported_item": ["import_clause"],
    },
    "typescript": {
        "import_node": ["import_statement", "lexical_declaration"],
        "get_imported_item": ["import_clause"],
    },
    "go": {
        "get_package_name": ["package_identifier"],
        "get_imported_package": ["import_spec"],
    },
    "java": {
        "get_package_name": ["package_declaration"],
        "get_imported_package": ["import_declaration"],
    },
    "python": {
        "get_package_name": ["module"],
        "get_imported_package": ["import_statement", "import_from_statement"],
    },
    "rust": {
        "package_node": ["mod_item"],
        "import_node": ["use_declaration"],
        "imported_item": ["use_clause", "scoped_identifier", "identifier", "use_list"],
    },
}


import_aliases_improved = {
    "javascript": {
        "package_node": ["program"],
        "import_node": [
            "import_statement",
            "lexical_declaration",
            "variable_declaration",
        ],
        "imported_item": ["import_clause", "named_imports", "identifier"],
    },
    "typescript": {
        "package_node": ["program"],
        "import_node": [
            "import_statement",
            "lexical_declaration",
            "variable_declaration",
        ],
        "imported_item": ["import_clause", "named_imports", "identifier"],
    },
    "go": {
        "package_node": ["package_clause", "package_identifier"],
        "import_node": ["import_declaration", "import_spec"],
        "imported_item": ["package_identifier", "import_spec"],
    },
    "java": {
        "package_node": ["package_declaration"],
        "import_node": ["import_declaration"],
        "imported_item": ["scoped_identifier"],
    },
    "python": {
        "package_node": ["module"],
        "import_node": ["import_statement", "import_from_statement"],
        "imported_item": ["dotted_name", "aliased_import", "identifier"],
    },
    "rust": {
        "package_node": ["mod_item"],
        "import_node": ["use_declaration"],
        "imported_item": ["use_clause", "scoped_identifier", "identifier", "use_list"],
    },
}

In [29]:
def _extract_js_ts_imports(parent_node: Node, source_bytes: bytes):
    chunks = []
    project_root = (
        "/home/user/Documents/Project_5/backend/experiments/repo/Testing"
    )
    for node in parent_node.children:
        # ----------------------------------------------------
        # 1. ES6 Imports: import { Car } from './Car.js'
        # ----------------------------------------------------
        if node.type == "import_statement":
            data = {}
            for child in node.children:
                if child.type == "string":
                    # Remove quotes: '"./Car.js"' -> './Car.js'
                    data["source"] = _text(child, source_bytes).strip("'\"`")
                    resolved_import = _resolve_import(
                        data["source"], str(file_path), project_root, "javascript"
                    )
                    data["file_type"] = resolved_import.get("category", "unkown")
                    data["resolved_path"] = resolved_import.get("resolved_path", None)
                elif child.type == "import_clause":
                    data["clause"] = [
                        item.strip()
                        for item in _text(child, source_bytes).split(",")
                        if item.strip()
                    ]

            # Default for side-effect imports like: import './setup.js'
            if "clause" not in data:
                data["clause"].extend("*")

            chunks.append(data)

        # ----------------------------------------------------
        # 2. CommonJS Require: const car = require('./Car.js')
        # ----------------------------------------------------
        elif node.type in ("lexical_declaration", "variable_declaration"):
            for declarator in node.children:
                if declarator.type != "variable_declarator":
                    continue

                value_node = declarator.child_by_field_name("value")
                if not value_node:
                    continue

                # Identify if value_node is direct require() or chained require().property
                req_call_node = None
                if value_node.type == "call_expression":
                    req_call_node = value_node
                elif value_node.type == "member_expression":
                    obj = value_node.child_by_field_name("object")
                    if obj and obj.type == "call_expression":
                        req_call_node = obj

                if not req_call_node:
                    continue

                # Ensure the function being called is actually 'require'
                func_node = req_call_node.child_by_field_name("function")
                if not func_node or _text(func_node, source_bytes) != "require":
                    continue

                # Extract the imported path safely from string node children
                args_node = req_call_node.child_by_field_name("arguments")
                if not args_node:
                    continue

                source = None
                for arg in args_node.children:
                    if arg.type == "string":
                        source = _text(arg, source_bytes).strip("'\"`")
                        resolved_import = _resolve_import(
                            source, str(file_path), project_root, "javascript"
                        )
                        category = resolved_import.get("category", "unkown")
                        resolved_path = resolved_import.get("resolved_path", None)
                        break

                if not source:
                    continue

                # Build data object only when a valid import is confirmed
                data = {
                    "source": source,
                    "file_type": category,
                    "resolved_path": resolved_path,
                    "clause": [],
                }
                name_node = declarator.child_by_field_name("name")

                if name_node:
                    if name_node.type == "object_pattern":
                        # Destructured: const { car, truck } = require(...)
                        raw_clause = _text(name_node, source_bytes)
                        data["clause"].extend(
                            [
                                item.strip()
                                for item in raw_clause.strip("{} \n\r\t").split(",")
                                if item.strip()
                            ]
                        )
                    else:
                        # Standard: const car = require(...)
                        data["clause"].extend(
                            [
                                item.strip()
                                for item in _text(child, source_bytes).split(",")
                                if item.strip()
                            ]
                        )
                else:
                    data["clause"].extend(["*"])

                chunks.append(data)

    return chunks

In [30]:
def _extract_js_ts_exports(parent_node: Node, source_bytes: bytes) -> list[dict]:
    exports = []

    for node in parent_node.children:
        # =====================================================================
        # 1. ESM EXPORTS: export const x = 1, export default X, export { x }
        # =====================================================================
        if node.type == "export_statement":
            # Check for re-export path: export { x } from './file.js' or export * from './file.js'
            source_file = None
            for child in node.children:
                if child.type == "string":
                    source_file = _text(child, source_bytes).strip("'\"`")

            # -----------------------------------------------------------------
            # Pattern A: Wildcard Re-export -> export * from './Car.js'
            # -----------------------------------------------------------------
            if any(child.type == "*" for child in node.children):
                exports.append({
                    "symbol": "*",
                    "type": "reexport_all",
                    "source": source_file
                })
                continue

            # -----------------------------------------------------------------
            # Pattern B: Export Clause -> export { car, truck as vehicle }
            # -----------------------------------------------------------------
            clause_node = None
            for child in node.children:
                if child.type == "export_clause":
                    clause_node = child
                    break

            if clause_node:
                for specifier in clause_node.children:
                    if specifier.type == "export_specifier":
                        # If alias exists: 'truck as vehicle', the exported name is 'vehicle' (alias)
                        alias_node = specifier.child_by_field_name("alias")
                        name_node = specifier.child_by_field_name("name")
                        
                        export_name = _text(alias_node or name_node, source_bytes)
                        exports.append({
                            "symbol": export_name,
                            "type": "reexport_named" if source_file else "named",
                            "source": source_file
                        })
                continue

            # -----------------------------------------------------------------
            # Pattern C: Declarations -> export const x = 1 / export function foo()
            # -----------------------------------------------------------------
            decl_node = node.child_by_field_name("declaration")
            if decl_node:
                # export const/let/var a = 1, b = 2
                if decl_node.type in ("lexical_declaration", "variable_declaration"):
                    for declarator in decl_node.children:
                        if declarator.type == "variable_declarator":
                            name_node = declarator.child_by_field_name("name")
                            if name_node:
                                if name_node.type == "object_pattern":
                                    # export const { car, truck } = obj
                                    for prop in name_node.children:
                                        if prop.type in ("shorthand_property_identifier_pattern", "property_identifier"):
                                            exports.append({"symbol": _text(prop, source_bytes), "type": "named", "source": None})
                                else:
                                    exports.append({"symbol": _text(name_node, source_bytes), "type": "named", "source": None})
                
                # export function foo() / export class Car
                elif decl_node.type in ("function_declaration", "class_declaration", "generator_function_declaration"):
                    name_node = decl_node.child_by_field_name("name")
                    if name_node:
                        exports.append({"symbol": _text(name_node, source_bytes), "type": "named", "source": None})
                continue

            # -----------------------------------------------------------------
            # Pattern D: Default Export -> export default Car / export default function()
            # -----------------------------------------------------------------
            if any(child.type == "default" for child in node.children):
                value_text = "default"
                value_node = node.child_by_field_name("value")
                if value_node and value_node.type == "identifier":
                    value_text = f"default ({_text(value_node, source_bytes)})"
                
                exports.append({
                    "symbol": value_text,
                    "type": "default",
                    "source": None
                })

        # =====================================================================
        # 2. COMMONJS EXPORTS: module.exports = ... or exports.car = ...
        # =====================================================================
        elif node.type == "expression_statement":
            expr = node.children[0] if node.children else None
            if not expr or expr.type != "assignment_expression":
                continue

            left_node = expr.child_by_field_name("left")
            right_node = expr.child_by_field_name("right")

            if not left_node:
                continue

            left_text = _text(left_node, source_bytes)

            # -----------------------------------------------------------------
            # Pattern A: module.exports = Car or module.exports = { car, truck }
            # -----------------------------------------------------------------
            if left_text == "module.exports":
                if right_node and right_node.type == "object":
                    # Object literal: module.exports = { car, truck: Vehicle }
                    for prop in right_node.children:
                        if prop.type in ("pair", "shorthand_property_identifier"):
                            key = prop.child_by_field_name("key") or prop
                            exports.append({
                                "symbol": _text(key, source_bytes),
                                "type": "named",
                                "source": None
                            })
                else:
                    # Single entity export: module.exports = Car
                    sym = _text(right_node, source_bytes) if right_node else "default"
                    exports.append({
                        "symbol": f"default ({sym})",
                        "type": "default",
                        "source": None
                    })

            # -----------------------------------------------------------------
            # Pattern B: module.exports.car = Car or exports.car = Car
            # -----------------------------------------------------------------
            elif left_text.startswith("module.exports.") or left_text.startswith("exports."):
                prop_name = left_text.split(".")[-1]
                exports.append({
                    "symbol": prop_name,
                    "type": "named",
                    "source": None
                })

    return exports

In [31]:
def _check_dynamic_import(call_node: Node):
    """Helper to find importlib.import_module(...) or __import__(...)"""
    project_root = (
        "/home/user/Documents/Project_5/backend/experiments/data/code/bank_py"
    )
    func_node = call_node.child_by_field_name("function")
    if not func_node:
        return None

    func_text = _text(func_node, source_bytes)

    # Check if it's importlib.import_module or __import__
    if func_text in ("importlib.import_module", "__import__"):
        args_node = call_node.child_by_field_name("arguments")
        if args_node and args_node.children:
            # Pehla argument package/module ka name hota hai
            for arg in args_node.children:
                if arg.type == "string":
                    # Quotes (' ' ya " ") hatana
                    mod_name = _text(arg, source_bytes).strip("'\"")
                    resolved_import = _resolve_import(
                        mod_name, str(file_path), project_root, "python"
                    )
                    file_type = resolved_import.get("category", "unknown")
                    resolved_path = resolved_import.get("resolved_path", None)
                    return {
                        "source": mod_name,
                        "clause": [],
                        "file_type": file_type,
                        "is_dynamic": True,
                        "resolved_path": resolved_path,
                    }
    return None


def _extract_py_imports(parent_node: Node, source_bytes: bytes):
    chunks = []
    project_root = (
        "/home/user/Documents/Project_5/backend/experiments/data/code/bank_py"
    )
    for node in parent_node.children:
        # Case 1: handles "from foo import bar, baz"
        if node.type == "import_from_statement":
            data = {"source": "", "clause": []}

            # Tree-sitter standard field for the module source
            module_node = node.child_by_field_name("module_name")
            if module_node:
                data["source"] = _text(module_node, source_bytes)
                resolved_import = _resolve_import(
                    data["source"], str(file_path), project_root, "python"
                )
                data["file_type"] = resolved_import.get("category", "unknown")
                data["resolved_path"] = resolved_import.get("resolved_path", None)

            # Extract imported items (names)
            for child in node.children:
                if child.type in ("dotted_name", "aliased_import", "identifier"):
                    # Avoid adding the module source itself to clauses
                    if child != module_node:
                        data["clause"].append(_text(child, source_bytes))
                elif child.type == "import_prefix":
                    # For relative imports like "from . import foo"
                    if not data["source"]:
                        data["source"] = _text(child, source_bytes)
                        resolved_import = _resolve_import(
                            data["source"], str(file_path), project_root, "python"
                        )
                        data["file_type"] = resolved_import.get("category", "unknown")
                        data["resolved_path"] = resolved_import.get(
                            "resolved_path", None
                        )

            chunks.append(data)

        # Case 2: handles "import foo, bar as b"
        elif node.type == "import_statement":
            data = {"source": None, "clause": []}

            for child in node.children:
                if child.type in ("dotted_name", "aliased_import"):
                    if child.type == "aliased_import":
                        for c in child.children:
                            if c.type == "dotted_name":
                                source = _text(c, source_bytes)
                                resolved_import = _resolve_import(
                                    source, str(file_path), project_root, "python"
                                )
                                data["file_type"] = resolved_import.get("category", "unknown")
                                data["resolved_path"] = resolved_import.get("resolved_path", None)
                    else:
                        source = _text(child, source_bytes) 
                        resolved_import = _resolve_import(
                            source, str(file_path), project_root, "python"
                        )
                        data["file_type"] = resolved_import.get("category", "unknown")
                        data["resolved_path"] = resolved_import.get("resolved_path", None)
                    data["clause"].append(_text(child, source_bytes))

            if data["clause"]:
                chunks.append(data)

        elif node.type == "call":
            dyn_data = _check_dynamic_import(node)
            if dyn_data:
                chunks.append(dyn_data)

    return chunks

In [32]:
def _extract_go_imports(parent_node: Node, source_bytes: bytes, context: str):
    imports = []

    for node in parent_node.children:
        if node.type == "import_declaration":
            # import_declaration ke andar single import_spec ya import_spec_list hota hai
            for child in node.children:
                if child.type == "import_spec_list":
                    # Grouped import block: import ( ... )
                    for spec in child.children:
                        if spec.type == "import_spec":
                            imports.append(_parse_go_import_spec(spec, source_bytes, context))

                elif child.type == "import_spec":
                    # Single import line: import "fmt"
                    imports.append(_parse_go_import_spec(child, source_bytes, context))

    return imports


def _parse_go_import_spec(spec_node: Node, source_bytes: bytes, context: str):
    # Tree-sitter Go grammar mein "path" aur "name" standard fields hote hain
    path_node = spec_node.child_by_field_name("path")
    name_node = spec_node.child_by_field_name("name")
    project_root = (
        "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_go"
    )

    # Path se surrounding double quotes (" ") hatane ke liye .strip('"')
    path = _text(path_node, source_bytes).strip('"`') if path_node else ""
    alias = _text(name_node, source_bytes) if name_node else None
    resolved_import = _resolve_import(path, str(file_path), project_root, "go", {"go_module_name": context})
    file_type = resolved_import.get("category", "unknown")
    resolved_path = resolved_import.get("resolved_path", None)
    return {
        "source": path,  # e.g., "math/rand" ya "github.com/lib/pq"
        "alias": alias,  # e.g., "m", "_", "." ya None (agar standard import ho)
        "file_type": file_type,
        "resolved_path": resolved_path,
    }

In [33]:
def _extract_java_imports_and_package(parent_node: Node, source_bytes: bytes):
    result = {"package": None, "imports": []}

    project_root = (
        "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_java"
    )

    # Java me top-level nodes direct parent_node (program) ke children hote hain
    for node in parent_node.children:

        # Case 1: Package declaration (e.g., "package com.example.project;")
        if node.type == "package_declaration":
            for child in node.children:
                if child.type in ("scoped_identifier", "identifier"):
                    result["package"] = _text(child, source_bytes)
                    break

        # Case 2: Import statements
        elif node.type == "import_declaration":
            import_info = {"path": "", "is_static": False, "is_wildcard": False}

            for child in node.children:
                # Static imports check (e.g., import static ...)
                if child.type == "static":
                    import_info["is_static"] = True

                # Main path/class identifier (e.g., java.util.List)
                elif child.type in ("scoped_identifier", "identifier"):
                    import_info["path"] = _text(child, source_bytes)
                    resolved_import = _resolve_import(
                        import_info["path"], str(file_path), project_root, "java"
                    )
                    import_info["file_type"] = resolved_import.get("category", "unknown")
                    import_info["resolved_path"] = resolved_import.get("resolved_path", None)

                # Wildcard imports check (e.g., import java.util.*)
                elif child.type == "asterisk":
                    import_info["is_wildcard"] = True

            # Agar wildcard star (*) tha to path me suffix append kar do
            if import_info["is_wildcard"] and import_info["path"]:
                resolved_import = _resolve_import(
                    import_info["path"], str(file_path), project_root, "java"
                )
                import_info["file_type"] = resolved_import.get("category", "unknown")
                import_info["resolved_path"] = resolved_import.get("resolved_path", None)
                import_info["path"] += ".*"

            if import_info["path"]:
                result["imports"].append(import_info)

    return result

In [34]:

def _flatten_use_tree(node: Node, prefix: str, source_bytes: bytes) -> list[str]:
    """
    Helper function: Handles Rust's nested use statements like `use std::{io, fs::File};`
    and flattens them into full paths.
    """
    results = []

    if node.type in ("identifier", "scoped_identifier"):
        text = _text(node, source_bytes)
        full_path = f"{prefix}::{text}" if prefix else text
        results.append(full_path)

    elif node.type in ("use_wildcard", "use_as_clause"):
        # Handles "*" wildcards and "as alias" imports
        text = _text(node, source_bytes)
        full_path = f"{prefix}::{text}" if prefix else text
        results.append(full_path)

    elif node.type == "scoped_use_list":
        # Handles patterns like `prefix::{ item1, item2 }`
        path_node = node.child_by_field_name("path")
        list_node = node.child_by_field_name("list")

        path_str = _text(path_node, source_bytes) if path_node else ""
        new_prefix = f"{prefix}::{path_str}" if prefix and path_str else (path_str or prefix)

        if list_node:
            results.extend(_flatten_use_tree(list_node, new_prefix, source_bytes))

    elif node.type == "use_list":
        # Iterates inside `{ ... }` blocks
        for child in node.children:
            if child.type not in (",", "{", "}"):
                results.extend(_flatten_use_tree(child, prefix, source_bytes))

    elif node.type == "self":
        # Handles `use std::io::{self, Read};` where `self` refers to `std::io`
        if prefix:
            results.append(prefix)

    return results


def _extract_rust_imports_and_modules(parent_node: Node, source_bytes: bytes):
    result = {
        "uses": [],       # Flattened use statements
        "modules": [],    # Mod declarations (mod foo;)
        "crates": []      # External crate statements
    }

    for node in parent_node.children:

        # Case 1: Use Declarations (use std::collections::HashMap;)
        if node.type == "use_declaration":
            for child in node.children:
                if child.type in ("scoped_identifier", "scoped_use_list", "use_wildcard", "use_as_clause", "identifier", "use_list"):
                    extracted = _flatten_use_tree(child, "", source_bytes)
                    result["uses"].extend(extracted)

        # Case 2: Module Declarations (mod utils; or pub mod internal { ... })
        elif node.type == "mod_item":
            name_node = node.child_by_field_name("name")
            body_node = node.child_by_field_name("body")

            if name_node:
                mod_info = {
                    "name": _text(name_node, source_bytes),
                    # True matlab file import (mod foo;), False matlab inline block (mod foo { ... })
                    "is_external": body_node is None  
                }
                result["modules"].append(mod_info)

        # Case 3: External Crates (extern crate serde;)
        elif node.type == "extern_crate_declaration":
            for child in node.children:
                if child.type == "identifier":
                    result["crates"].append(_text(child, source_bytes))
                    break

    return result

In [35]:
# python
# file_path = Path("/home/user/Documents/Project_5/backend/experiments/data/code/bank_py/main.py")
# go
file_path = Path(
    "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_go/main.go"
)
go_root_path = Path("/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_go")

# java
# file_path = Path(
#     "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_java/src/com/example/vehicle/Main.java"
# )

#js
# file_path = Path("/home/user/Documents/Project_5/backend/experiments/repo/Testing/backend/src/index.ts")

tree = parse_ast(file_path)
lang_name = extension_to_language_name(file_path)
source_bytes = file_path.read_bytes()

context = _get_go_module_name(go_root_path)


if tree:

    chunk = _extract_go_imports(tree.root_node, source_bytes, context)
    print(chunk)

go
<Language id=131426052457824, version=14, name=None>


NameError: name '_get_go_module_name' is not defined

In [ ]:
[
    {
        "clause": ["express"],
        "source": "express",
        "file_type": "external",
        "resolved_path": None,
    },
    {
        "clause": ["{ connectToDB }"],
        "source": "./config/db",
        "file_type": "internal",
        "resolved_path": "/home/user/Documents/Project_5/backend/experiments/repo/Testing/backend/src/config/db.ts",
    },
    {
        "clause": ["router"],
        "source": "./routes/index",
        "file_type": "internal",
        "resolved_path": "/home/user/Documents/Project_5/backend/experiments/repo/Testing/backend/src/routes/index.ts",
    },
    {
        "clause": ["cookieParser"],
        "source": "cookie-parser",
        "file_type": "external",
        "resolved_path": None,
    },
    {
        "clause": ["initSocket"],
        "source": "./sockets",
        "file_type": "internal",
        "resolved_path": "/home/user/Documents/Project_5/backend/experiments/repo/Testing/backend/src/sockets/index.ts",
    },
    {
        "clause": ["{ createServer }"],
        "source": "node:http",
        "file_type": "stdlib",
        "resolved_path": None,
    },
    {
        "clause": ["{ Server }"],
        "source": "socket.io",
        "file_type": "external",
        "resolved_path": None,
    },
    {
        "clause": ["cors"],
        "source": "cors",
        "file_type": "external",
        "resolved_path": None,
    },
    {
        "clause": ["path"],
        "source": "path",
        "file_type": "stdlib",
        "resolved_path": None,
    },
]

{'package': 'com.example.app',
 'imports': [{'path': 'com.example.vehicle.*',
   'is_static': False,
   'is_wildcard': True,
   'file_type': 'internal',
   'resolved_path': '/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_java/src/com/example/vehicle'},
  {'path': 'com.example.vehicle.Car',
   'is_static': False,
   'is_wildcard': False,
   'file_type': 'internal',
   'resolved_path': '/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_java/src/com/example/vehicle/Car.java'},
  {'path': 'java.lang.Math.PI',
   'is_static': True,
   'is_wildcard': False,
   'file_type': 'stdlib',
   'resolved_path': None},
  {'path': 'java.util.List',
   'is_static': False,
   'is_wildcard': False,
   'file_type': 'stdlib',
   'resolved_path': None}]}

In [ ]:
from tree_sitter import Node, Query, QueryCursor

def extract_connections(tree, query_scm_string: str, source_bytes: bytes) -> list[dict]:
    language = tree.language
    query = Query(language, query_scm_string)

    
    # 1. Pass ONLY query to QueryCursor
    cursor = QueryCursor(query)
    
    # 2. Pass tree.root_node HERE inside captures()
    captures_dict = cursor.captures(tree.root_node)
    
    results = []
    # captures_dict is: {"capture_name": [node1, node2, ...]}
    for capture_name, nodes in captures_dict.items():
        if "." not in capture_name:
            continue
            
        kind, field = capture_name.split(".", 1)
        
        for node in nodes:
            text_val = source_bytes[node.start_byte:node.end_byte].decode("utf-8").strip("'\"`")
            results.append({
                "kind": kind,       # "import" or "export"
                "field": field,     # "path", "symbol", "alias", etc.
                "value": text_val
            })
            
    return results

In [ ]:
file_path = Path("/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_go/main.go")
tree = parse_ast(file_path)
lang_name = extension_to_language_name(file_path)
source_bytes = file_path.read_bytes()

query_scm_string = Path("/home/user/Documents/Project_5/backend/experiments/data/queries/python.scm").read_text()

if tree:
    chunk = extract_connections(tree, query_scm_string, source_bytes)
    print(chunk)

QueryError: Invalid node type at row 1, column 1: import_statement

In [ ]:
def extract_imports(node: Node, language: str):
    get_alias_langauge = import_aliases.get(language)
    if not get_alias_langauge:
        return None
    for node in node.children:
        pass

In [ ]:
# imports.py
# Field-based import extraction. Each function returns a list of
# {"module": str, "alias": str|None, "is_named": bool} dicts.

def _text(node, source_bytes):
    return source_bytes[node.start_byte:node.end_byte].decode("utf-8")


def extract_python_imports(node, source_bytes, results=None):
    if results is None:
        results = []
    if node.type == "import_statement":
        for child in node.children:
            if child.type == "dotted_name":
                results.append({"module": _text(child, source_bytes), "alias": None})
            elif child.type == "aliased_import":
                name_node = child.child_by_field_name("name")
                alias_node = child.child_by_field_name("alias")
                results.append({
                    "module": _text(name_node, source_bytes) if name_node else None,
                    "alias": _text(alias_node, source_bytes) if alias_node else None,
                })
    elif node.type == "import_from_statement":
        module_node = node.child_by_field_name("module_name")
        module = _text(module_node, source_bytes) if module_node else None
        results.append({"module": module, "alias": None, "is_named": True})
    for child in node.children:
        extract_python_imports(child, source_bytes, results)
    return results


def extract_js_ts_imports(node, source_bytes, results=None):
    if results is None:
        results = []
    if node.type == "import_statement":
        source_node = node.child_by_field_name("source")   # the string literal
        if source_node:
            module = _text(source_node, source_bytes).strip("'\"")
            results.append({"module": module, "alias": None})
    # CommonJS: const x = require('y')  -- a *specific pattern*, not a generic type scan
    elif node.type == "variable_declarator":
        value = node.child_by_field_name("value")
        if value and value.type == "call_expression":
            fn = value.child_by_field_name("function")
            args = value.child_by_field_name("arguments")
            if fn and _text(fn, source_bytes) == "require" and args:
                for arg in args.children:
                    if arg.type == "string":
                        module = _text(arg, source_bytes).strip("'\"")
                        name_node = node.child_by_field_name("name")
                        results.append({
                            "module": module,
                            "alias": _text(name_node, source_bytes) if name_node else None,
                        })
    for child in node.children:
        extract_js_ts_imports(child, source_bytes, results)
    return results


def extract_go_imports(node, source_bytes, results=None):
    if results is None:
        results = []
    if node.type == "import_spec":
        path_node = node.child_by_field_name("path")
        name_node = node.child_by_field_name("name")   # alias, if present
        if path_node:
            results.append({
                "module": _text(path_node, source_bytes).strip('"'),
                "alias": _text(name_node, source_bytes) if name_node else None,
            })
    for child in node.children:
        extract_go_imports(child, source_bytes, results)
    return results


def extract_java_imports(node, source_bytes, results=None):
    if results is None:
        results = []
    if node.type == "import_declaration":
        for child in node.children:
            if child.type in ("scoped_identifier", "identifier"):
                results.append({"module": _text(child, source_bytes), "alias": None})
    for child in node.children:
        extract_java_imports(child, source_bytes, results)
    return results


def extract_rust_imports(node, source_bytes, results=None):
    if results is None:
        results = []
    if node.type == "use_declaration":
        arg = node.child_by_field_name("argument")
        if arg:
            results.append({"module": _text(arg, source_bytes), "alias": None})
    for child in node.children:
        extract_rust_imports(child, source_bytes, results)
    return results


EXTRACTORS = {
    "python": extract_python_imports,
    "javascript": extract_js_ts_imports,
    "typescript": extract_js_ts_imports,
    "go": extract_go_imports,
    "java": extract_java_imports,
    "rust": extract_rust_imports,
}


def resolve_import(module_path: str, current_file: str, repo_files: set, lang: str):
    """Best-effort: is this import internal (exists in repo) or external?
    Not a full reimplementation of each language's resolver -- just enough
    to classify the edge. Extend per-language as real gaps show up."""
    import os

    if lang in ("javascript", "typescript"):
        if module_path.startswith("."):
            base = os.path.normpath(os.path.join(os.path.dirname(current_file), module_path))
            for suffix in ("", ".js", ".ts", ".jsx", ".tsx", "/index.js", "/index.ts"):
                if base + suffix in repo_files:
                    return {"internal": True, "resolved_path": base + suffix}
        return {"internal": False, "resolved_path": None}

    if lang == "python":
        candidate = module_path.replace(".", "/")
        for suffix in (".py", "/__init__.py"):
            if candidate + suffix in repo_files:
                return {"internal": True, "resolved_path": candidate + suffix}
        return {"internal": False, "resolved_path": None}

    # go/java/rust: match by trailing path segment as a cheap heuristic
    tail = module_path.split("/")[-1].split(".")[-1]
    for f in repo_files:
        if f.endswith(tail):
            return {"internal": True, "resolved_path": f}
    return {"internal": False, "resolved_path": None}

In [ ]:
import os
from pathlib import Path
import sys

# Cache for go.mod module names to avoid re-reading disk continuously
_GO_MODULE_CACHE: dict[str, str | None] = {}


def _get_go_module_name(project_root_path: Path) -> str | None:
    """Reads the module path from go.mod if present."""
    root_str = str(project_root_path)
    if root_str in _GO_MODULE_CACHE:
        return _GO_MODULE_CACHE[root_str]

    go_mod_file = project_root_path / "go.mod"
    module_name = None
    if go_mod_file.exists():
        for line in go_mod_file.read_text(
            encoding="utf-8", errors="ignore"
        ).splitlines():
            line = line.strip()
            if line.startswith("module "):
                module_name = line.split()[1].strip("'\"")
                break

    _GO_MODULE_CACHE[root_str] = module_name
    return module_name


def classify_import(
    import_path: str, current_file_path: str, project_root: str, language: str
) -> str:
    project_root_path = Path(project_root).resolve()
    current_dir = Path(current_file_path).parent.resolve()

    # Clean up input string
    import_path = import_path.strip("'\"`;")

    # -------------------------------------------------------------------------
    # 1. GO
    # -------------------------------------------------------------------------
    if language == "go":
        # Relative imports (rare in modern Go, but exists in legacy code)
        if import_path.startswith("./") or import_path.startswith("../"):
            return "internal"

        go_module_name = _get_go_module_name(project_root_path)

        # Internal: Starts with project's go.mod module name
        # e.g., import "github.com/myorg/myproject/pkg/util"
        if go_module_name and import_path.startswith(go_module_name):
            return "internal"

        # Stdlib vs External:
        # Standard library packages in Go NEVER have a dot in the first path segment
        # e.g. "fmt", "net/http", "math/rand" -> stdlib
        # e.g. "github.com/gin-gonic/gin", "golang.org/x/sync" -> external
        first_segment = import_path.split("/")[0]
        if "." not in first_segment:
            return "stdlib"

        return "external"

    # -------------------------------------------------------------------------
    # 2. JAVA
    # -------------------------------------------------------------------------
    elif language == "java":
        # Standard Java Runtime Library Prefixes
        JAVA_STDLIB_PREFIXES = (
            "java.",
            "javax.",
            "jdk.",
            "sun.",
            "com.sun.",
            "org.w3c.dom",
            "org.xml.sax",
        )
        if any(import_path.startswith(prefix) for prefix in JAVA_STDLIB_PREFIXES):
            return "stdlib"

        # Wildcard imports or symbol imports -> convert package to file path
        # e.g., "com.company.app.models.Car" -> "com/company/app/models/Car.java"
        # e.g., "com.company.app.models.*" -> "com/company/app/models"
        clean_path = import_path.removesuffix(".*")
        rel_file_path = clean_path.replace(".", "/") + ".java"
        rel_dir_path = clean_path.replace(".", "/")

        # Check standard Java source directory layouts
        possible_roots = [
            project_root_path,
            project_root_path / "src" / "main" / "java",
            project_root_path / "src" / "test" / "java",
            project_root_path / "src",
        ]

        for root in possible_roots:
            target_file = root / rel_file_path
            target_dir = root / rel_dir_path
            if target_file.exists() or (target_dir.exists() and target_dir.is_dir()):
                return "internal"

        return "external"

    if language in ("javascript", "typescript"):
        # Built-in Node modules
        if import_path.startswith("node:") or import_path in (
            "fs",
            "path",
            "http",
            "crypto",
            "os",
        ):
            return "stdlib"

        # Explicit relative paths: ./Car or ../utils
        if import_path.startswith("./") or import_path.startswith("../"):
            return "internal"

        # Path Aliases (e.g. "@/components/Car" -> project_root/src/components/Car)
        if import_path.startswith("@/") or import_path.startswith("~"):
            return "internal"

        # Bare imports like 'react', 'lodash', 'express'
        return "external"

    # -------------------------------------------------------------------------
    # 2. PYTHON
    # -------------------------------------------------------------------------
    elif language == "python":
        # Check standard library modules
        if import_path.split(".")[0] in sys.stdlib_module_names:
            return "stdlib"

        # Relative imports: from . import car or from ..utils import math
        if import_path.startswith("."):
            return "internal"

        # Check if absolute import maps to a local file/folder in project root
        # e.g., 'import my_module' -> check if project_root/my_module.py exists
        top_level_module = import_path.split(".")[0]
        possible_py_file = project_root_path / f"{top_level_module}.py"
        possible_py_dir = project_root_path / top_level_module

        if possible_py_file.exists() or (
            possible_py_dir.exists() and possible_py_dir.is_dir()
        ):
            return "internal"

        return "external"

    # -------------------------------------------------------------------------
    # 3. RUST
    # -------------------------------------------------------------------------
    elif language == "rust":
        if (
            import_path.startswith("std::")
            or import_path.startswith("core::")
            or import_path.startswith("alloc::")
        ):
            return "stdlib"

        if any(
            import_path.startswith(prefix)
            for prefix in ("crate::", "super::", "self::")
        ):
            return "internal"

        return "external"

    # Fallback default
    return "external"

In [ ]:
import sys
from pathlib import Path

NODE_BUILTINS = {
    "assert",
    "buffer",
    "child_process",
    "cluster",
    "crypto",
    "dgram",
    "dns",
    "events",
    "fs",
    "http",
    "http2",
    "https",
    "net",
    "os",
    "path",
    "perf_hooks",
    "process",
    "querystring",
    "readline",
    "stream",
    "string_decoder",
    "timers",
    "tls",
    "tty",
    "url",
    "util",
    "v8",
    "vm",
    "worker_threads",
    "zlib",
}

JAVA_STDLIB_PREFIXES = (
    "java.",
    "javax.",
    "jdk.",
    "sun.",
    "com.sun.",
    "org.w3c.dom",
    "org.xml.sax",
)

JS_TS_SUFFIXES = (
    "",
    ".ts",
    ".tsx",
    ".js",
    ".jsx",
    ".mjs",
    ".cjs",
    "/index.ts",
    "/index.tsx",
    "/index.js",
    "/index.jsx",
)

STDLIB_MODULES = getattr(sys, "stdlib_module_names", None) or set(
    sys.builtin_module_names
)


def _resolve_import(
    import_path: str,
    current_file_path: str,
    project_root: str,
    language: str,
    context: dict | None = None,
) -> dict:
    """
    Returns {"category": "stdlib" | "internal" | "external", "resolved_path": str | None}

    `context` carries per-repo data computed ONCE per indexing job (not per call):
      - context["go_module_name"]: str, from go.mod
      - context["rust_crates"]: dict[str, str] mapping crate name -> crate root dir,
                                 from workspace Cargo.toml
    Pass these in rather than re-reading go.mod/Cargo.toml on every import.
    """
    context = context or {}
    project_root_path = Path(project_root).resolve()
    current_dir = Path(current_file_path).parent.resolve()
    import_path = import_path.strip("'\"`;")

    # -------------------------------------------------------------------
    # GO
    # -------------------------------------------------------------------
    if language == "go":
        if import_path.startswith("./") or import_path.startswith("../"):
            resolved = (current_dir / import_path).resolve()
            return {"category": "internal", "resolved_path": str(resolved)}

        go_module_name = context.get("go_module_name")
        if go_module_name and import_path.startswith(go_module_name):
            # Go imports resolve to a PACKAGE (directory), not a single file --
            # a Go package is typically many files sharing one directory.
            sub_path = import_path[len(go_module_name) :].lstrip("/")
            resolved_dir = project_root_path / sub_path
            return {
                "category": "internal",
                "resolved_path": str(resolved_dir) if resolved_dir.exists() else None,
            }

        first_segment = import_path.split("/")[0]
        if "." not in first_segment:
            return {"category": "stdlib", "resolved_path": None}
        return {"category": "external", "resolved_path": None}

    # -------------------------------------------------------------------
    # JAVA
    # -------------------------------------------------------------------
    elif language == "java":
        if any(import_path.startswith(p) for p in JAVA_STDLIB_PREFIXES):
            return {"category": "stdlib", "resolved_path": None}

        # Wildcard imports ke liye .* remove karein
        clean_path = import_path.removesuffix(".*")
        parts = clean_path.split(".")

        # Candidates generate karein taaki static imports aur inner classes handle ho sakein
        # Example: com.example.Car.MAX_SPEED -> pehle 'com/example/Car/MAX_SPEED.java' check karega,
        # fir fallback karke 'com/example/Car.java' check karega.
        path_candidates = []
        for i in range(len(parts), 0, -1):
            path_candidates.append("/".join(parts[:i]))

        for base_path in path_candidates:
            rel_file_path = base_path + ".java"
            rel_dir_path = base_path

            # 1. Fast direct checks in common roots (Flat layout, src, and Maven/Gradle layouts)
            for root_prefix in ["", "src", "src/main/java", "src/test/java"]:
                root_path = (
                    project_root_path / root_prefix
                    if root_prefix
                    else project_root_path
                )

                # Check for exact file
                candidate_file = root_path / rel_file_path
                if candidate_file.exists() and candidate_file.is_file():
                    return {
                        "category": "internal",
                        "resolved_path": str(candidate_file),
                    }

                # Check for directory (for wildcard package imports like com.example.vehicle.*)
                candidate_dir = root_path / rel_dir_path
                if candidate_dir.exists() and candidate_dir.is_dir():
                    return {"category": "internal", "resolved_path": str(candidate_dir)}

            # 2. Multi-module Maven/Gradle layouts using globbing
            for pattern_root in ("src/main/java", "src/test/java"):
                matches = list(
                    project_root_path.glob(f"**/{pattern_root}/{rel_file_path}")
                )
                if matches:
                    return {"category": "internal", "resolved_path": str(matches[0])}
                dir_matches = list(
                    project_root_path.glob(f"**/{pattern_root}/{rel_dir_path}")
                )
                if dir_matches and dir_matches[0].is_dir():
                    return {
                        "category": "internal",
                        "resolved_path": str(dir_matches[0]),
                    }

        return {"category": "external", "resolved_path": None}
    # -------------------------------------------------------------------
    # JAVASCRIPT / TYPESCRIPT
    # -------------------------------------------------------------------
    elif language in ("javascript", "typescript"):
        if (
            import_path.startswith("node:")
            or import_path.split("/")[0] in NODE_BUILTINS
        ):
            return {"category": "stdlib", "resolved_path": None}

        if import_path.startswith("./") or import_path.startswith("../"):
            base = (current_dir / import_path).resolve()
            for suffix in JS_TS_SUFFIXES:
                if suffix.startswith("/"):
                    candidate = base / suffix.lstrip("/")
                else:
                    candidate = Path(str(base) + suffix)
                if candidate.exists() and candidate.is_file():
                    return {"category": "internal", "resolved_path": str(candidate)}
            return {
                "category": "internal",
                "resolved_path": None,
            }  # relative but unresolved

        # Common alias convention (Next.js/Vue default) -- best-effort, not tsconfig-aware
        if import_path.startswith("@/") or import_path.startswith("~/"):
            rel = import_path.split("/", 1)[1]
            base = project_root_path / "src" / rel
            for suffix in JS_TS_SUFFIXES:
                if suffix.startswith("/"):
                    candidate = base / suffix.lstrip("/")
                else:
                    candidate = Path(str(base) + suffix)
                if candidate.exists() and candidate.is_file():
                    return {"category": "internal", "resolved_path": str(candidate)}
            return {"category": "internal", "resolved_path": None}

        return {"category": "external", "resolved_path": None}

    # -------------------------------------------------------------------
    # PYTHON
    # -------------------------------------------------------------------
    elif language == "python":
        if import_path.split(".")[0] in STDLIB_MODULES:
            return {"category": "stdlib", "resolved_path": None}

        if import_path.startswith("."):
            # count leading dots = how many dirs up; strip them for the module part
            level = len(import_path) - len(import_path.lstrip("."))
            module_part = import_path.lstrip(".")
            base_dir = current_dir
            for _ in range(level - 1):
                base_dir = base_dir.parent
            if module_part:
                candidate_file = base_dir / (module_part.replace(".", "/") + ".py")
                candidate_pkg = base_dir / module_part.replace(".", "/") / "__init__.py"
            else:
                candidate_file = None
                candidate_pkg = base_dir / "__init__.py"
            for c in (candidate_file, candidate_pkg):
                if c and c.exists():
                    return {"category": "internal", "resolved_path": str(c)}
            return {"category": "internal", "resolved_path": None}

        top_level_module = import_path.split(".")[0]
        rest = import_path.split(".")[1:]
        for src_root in (project_root_path, project_root_path / "src"):
            candidate_file = src_root / top_level_module
            for part in rest:
                candidate_file = candidate_file / part
            py_file = Path(str(candidate_file) + ".py")
            init_file = candidate_file / "__init__.py"
            if py_file.exists():
                return {"category": "internal", "resolved_path": str(py_file)}
            if init_file.exists():
                return {"category": "internal", "resolved_path": str(init_file)}

        return {"category": "external", "resolved_path": None}

    # -------------------------------------------------------------------
    # RUST
    # -------------------------------------------------------------------
    elif language == "rust":
        if import_path.startswith(("std::", "core::", "alloc::")):
            return {"category": "stdlib", "resolved_path": None}

        if import_path.startswith(("crate::", "super::", "self::")):
            # resolvable in principle by walking module tree from current file;
            # left unresolved here (best-effort) unless you build the mod-tree mapper
            return {"category": "internal", "resolved_path": None}

        first_segment = import_path.split("::")[0]
        rust_crates = context.get("rust_crates", {})
        if first_segment in rust_crates:
            return {"category": "internal", "resolved_path": rust_crates[first_segment]}

        return {"category": "external", "resolved_path": None}

    return {"category": "external", "resolved_path": None}

In [ ]:
path = "/home/user/Documents/Project_5/backend/experiments/data/code/bank_py/main.py"
file_path = Path(f"{path}/main.py")
tree = parse_ast(Path(path))
lang_name = extension_to_language_name(Path(path))
if tree:
    imports = extract_python_imports(tree.root_node, Path(path).read_bytes())
    print(imports)
    if imports:
        for i in imports:
            repo_root = str(Path(path).parent)
            repo_files = {str(p) for p in Path(repo_root).rglob("*") if p.is_file()}
            resolved = resolve_import(
                i.get("module"),
                path,
                repo_files,
                lang_name,
            )
            print(resolved)

[{'module': 'savings_account', 'alias': None, 'is_named': True}, {'module': 'checking_account', 'alias': None, 'is_named': True}, {'module': 'os', 'alias': None, 'is_named': True}, {'module': 'math', 'alias': 'm'}, {'module': 'os', 'alias': None}, {'module': 'bank_account', 'alias': None}, {'module': 'http', 'alias': None}, {'module': 'fastapi', 'alias': None, 'is_named': True}]
{'internal': False, 'resolved_path': None}
{'internal': False, 'resolved_path': None}
{'internal': False, 'resolved_path': None}
{'internal': False, 'resolved_path': None}
{'internal': False, 'resolved_path': None}
{'internal': False, 'resolved_path': None}
{'internal': False, 'resolved_path': None}
{'internal': False, 'resolved_path': None}


In [ ]:
[
    {"module": "./Vehicle.js", "alias": None},
    {"module": "./Car.js", "alias": None},
    {"module": "./ElectricCar.js", "alias": None},
    {"module": "react", "alias": None},
    {"module": "node:fs", "alias": None},
    {"module": "./Car.js", "alias": "car"},
]
{
    "internal": True,
    "resolved_path": "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_js/Vehicle.js",
}
{
    "internal": True,
    "resolved_path": "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_js/Car.js",
}
{
    "internal": True,
    "resolved_path": "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_js/ElectricCar.js",
}
{"internal": False, "resolved_path": None}
{"internal": False, "resolved_path": None}
{
    "internal": True,
    "resolved_path": "/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_js/Car.js",
}

{'internal': True,
 'resolved_path': '/home/user/Documents/Project_5/backend/experiments/data/code/Vehicle_js/Car.js'}